In [ ]:
# install des trucs qu'il faut pour le textblob
from textblob import TextBlob
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
! python -m textblob.download_corpora

In [ ]:
import pandas as pd
import re
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from textblob import TextBlob, Word
from wordcloud import WordCloud
from nltk.corpus import stopwords

# chargement du dataset amazon
df = pd.read_csv("../datasets/amazon.csv")
df

1. Faire une analyse de sentiment sur les reviews, classer chaque review en positif/négatif/neutre, puis afficher un bar chart de la distribution.

In [ ]:
# on recup la colonne texte
col_texte = 'reviewText' if 'reviewText' in df.columns else df.select_dtypes(include='object').columns[0]
reviews = df[col_texte].fillna('').astype(str)
reviews = reviews[reviews.str.strip() != '']  # on vire les vides

# calcul de la polarité pour chaque review
polarites = reviews.apply(lambda t: TextBlob(t).sentiment.polarity)

# classification : positif si > 0.1, negatif si < -0.1, sinon neutre
sentiments = polarites.apply(
    lambda p: 'positive' if p > 0.1 else ('negative' if p < -0.1 else 'neutral')
)

nb_sentiments = Counter(sentiments)

# graphique bar
plt.figure(figsize=(8, 6))
couleurs = {'positive': '#2ecc71', 'negative': '#e74c3c', 'neutral': '#3498db'}
x_vals = list(nb_sentiments.keys())
y_vals = list(nb_sentiments.values())
bar_colors = [couleurs.get(s, 'gray') for s in x_vals]
plt.bar(x_vals, y_vals, color=bar_colors)
plt.title('Distribution des sentiments')
plt.xlabel('Sentiment')
plt.ylabel('Nombre de reviews')
plt.show()

print('Colonne utilisée :', col_texte)
print('Nombre de reviews analysées :', len(reviews))
print('Résultats :', dict(nb_sentiments))

2. Nettoyer le texte : mettre en minuscules, enlever la ponctuation et les chiffres, retirer les stopwords, puis lemmatiser avec les tags POS.

In [ ]:
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)

mots_vides = set(stopwords.words('english'))

# fonction pour convertir le tag POS en format wordnet
def pos_vers_wordnet(tag):
    if tag.startswith('J'):
        return 'a'  # adjectif
    elif tag.startswith('V'):
        return 'v'  # verbe
    elif tag.startswith('N'):
        return 'n'  # nom
    elif tag.startswith('R'):
        return 'r'  # adverbe
    return 'n'  # par defaut nom

tous_lemmes = []
tous_tags = []

# boucle sur chaque review
for texte in reviews:
    # nettoyage : minuscules + que des lettres
    propre = re.sub(r'[^a-zA-Z\s]', ' ', texte.lower())
    propre = re.sub(r'\s+', ' ', propre).strip()
    if propre == '':
        continue

    # tag POS puis lemmatisation
    for mot, tag in TextBlob(propre).tags:
        if mot in mots_vides or len(mot) <= 1:
            continue
        lemme = Word(mot).lemmatize(pos_vers_wordnet(tag))
        tous_lemmes.append(lemme)
        tous_tags.append(tag)

print('Nombre de tokens lemmatisés :', len(tous_lemmes))

3. Calculer la fréquence de chaque mot lemmatisé et afficher les 15 mots les plus fréquents en bar chart.

In [ ]:
# comptage des mots
compteur_mots = Counter(tous_lemmes)
top_15 = compteur_mots.most_common(15)

if top_15:
    mots, frequences = zip(*top_15)
    plt.figure(figsize=(12, 6))
    plt.barh(range(len(mots)), list(frequences), color='#2c3e50')
    plt.yticks(range(len(mots)), list(mots))
    plt.gca().invert_yaxis()  # le plus frequent en haut
    plt.title('Top 15 des mots les plus fréquents (après lemmatisation)')
    plt.xlabel('Fréquence')
    plt.ylabel('Mot')
    plt.tight_layout()
    plt.show()
else:
    print('Aucun mot trouvé après le prétraitement.')

4. Générer un nuage de mots (word cloud) à partir de tous les mots lemmatisés.

In [ ]:
# on joint tous les lemmes en un seul texte
corpus_complet = ' '.join(tous_lemmes)

if corpus_complet.strip():
    nuage = WordCloud(
        width=900, height=450,
        background_color='white',
        colormap='viridis',
        max_words=150
    ).generate(corpus_complet)

    plt.figure(figsize=(12, 6))
    plt.imshow(nuage, interpolation='bilinear')
    plt.axis('off')
    plt.title('Nuage de mots des reviews')
    plt.show()
else:
    print('Pas de mots pour le nuage.')

5. Visualiser la fréquence des noms, verbes et adjectifs dans le texte.

In [ ]:
# comptage par catégorie grammaticale
categories = {'Noms': 0, 'Verbes': 0, 'Adjectifs': 0}
for tag in tous_tags:
    if tag.startswith('N'):
        categories['Noms'] += 1
    elif tag.startswith('V'):
        categories['Verbes'] += 1
    elif tag.startswith('J'):
        categories['Adjectifs'] += 1

plt.figure(figsize=(8, 5))
noms_cat = list(categories.keys())
valeurs_cat = list(categories.values())
palette_couleurs = ['#3498db', '#e67e22', '#9b59b6']
barres = plt.bar(noms_cat, valeurs_cat, color=palette_couleurs)

# ajout des valeurs sur les barres
for b in barres:
    hauteur = b.get_height()
    plt.text(b.get_x() + b.get_width()/2, hauteur + 500,
             f'{hauteur:,}', ha='center', fontsize=11)

plt.title('Fréquence des noms, verbes et adjectifs')
plt.xlabel('Catégorie grammaticale')
plt.ylabel('Nombre')
plt.show()